In [1]:
import numpy as np  
import pandas as pd  
from IPython.display import display, HTML 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/movie-recommendation-system/movies.csv
/kaggle/input/movie-recommendation-system/ratings.csv


In [2]:
movies_df = pd.read_csv('/kaggle/input/movie-recommendation-system/movies.csv')
ratings_df = pd.read_csv('/kaggle/input/movie-recommendation-system/ratings.csv')
 
movies_columns = movies_df.columns.tolist()
ratings_columns = ratings_df.columns.tolist()
 
max_len = max(len(movies_columns), len(ratings_columns))
 
movies_columns_extended = movies_columns + [''] * (max_len - len(movies_columns))
ratings_columns_extended = ratings_columns + [''] * (max_len - len(ratings_columns))
 
columns_df = pd.DataFrame({ "movies": movies_columns_extended, "ratings": ratings_columns_extended })

display(HTML(columns_df.to_html(index=False)))

movies,ratings
movieId,userId
title,movieId
genres,rating
,timestamp


In [3]:
import pandas as pd
from IPython.display import HTML, display
 
movies_and_ratings = pd.merge(ratings_df, movies_df, on='movieId')
movies_and_ratings['timestamp'] = pd.to_datetime(movies_and_ratings['timestamp'], unit='s')
 
table1 = movies_and_ratings.head(5).to_html(index=False)

#######################################################################################################

missing_percentages = movies_and_ratings.isnull().mean() * 100
missing_df = pd.DataFrame({
    "Column": missing_percentages.index,
    "Percent Missing": missing_percentages.values
})
table2 = missing_df.to_html(index=False)

#######################################################################################################

combined_html = f"""
<table style="width:100%">
  <tr>
    <td style="vertical-align:top; width:50%">{table1}</td>
    <td style="vertical-align:top; width:50%">{table2}</td>
  </tr>
</table>
"""

display(HTML(combined_html))


In [4]:
import pandas as pd
from IPython.display import HTML, display
 
######################################################################################################

rating_counts = movies_and_ratings['rating'].value_counts().reset_index()
rating_counts.columns = ['Rating', 'Count']
rating_counts = rating_counts.sort_values(by='Count', ascending=False)
total_ratings = movies_and_ratings['rating'].count()
rating_counts['Percentage'] = rating_counts['Count'] / total_ratings * 100
rating_counts['Count'] = rating_counts['Count'].apply(lambda x: f"{x:,}")
rating_counts['Percentage'] = rating_counts['Percentage'].apply(lambda x: f"{x:.1f}%")
table1 = rating_counts.to_html(index=False)

######################################################################################################

top_movies = movies_and_ratings[movies_and_ratings['rating'] == 5] \
                .groupby('title')['rating'] \
                .count() \
                .reset_index()
top_movies.columns = ['Title', 'Count']
top_movies['Rating'] = 5
top_movies = top_movies[['Title', 'Rating', 'Count']]
top_movies = top_movies.sort_values(by='Count', ascending=False)
top_movies['Count'] = top_movies['Count'].apply(lambda x: f"{x:,}")
table2 = top_movies.head(10).to_html(index=False)

######################################################################################################

unique_movies = movies_and_ratings[['movieId', 'genres']].drop_duplicates()
genre_counts = unique_movies['genres'].value_counts().reset_index()
genre_counts.columns = ['Genre Combination', 'Count']
genre_counts['Count'] = genre_counts['Count'].apply(lambda x: f"{x:,}")
table3 = genre_counts.head(10).to_html(index=False)

######################################################################################################

combined_html = f"""
<div style="display: flex; gap: 100px; flex-wrap: wrap; align-items: flex-start;">
  <div style="flex: none; margin: 0;">{table1}</div>
  <div style="flex: none; margin: 0;">{table2}</div>
  <div style="flex: none; margin: 0;">{table3}</div>
</div>
"""

display(HTML(combined_html))


Rating,Count,Percentage
4.0,"6,639,798",26.6%
3.0,"4,896,928",19.6%
5.0,"3,612,474",14.4%
3.5,"3,177,318",12.7%
4.5,"2,200,539",8.8%
2.0,"1,640,868",6.6%
2.5,"1,262,797",5.1%
1.0,"776,815",3.1%
1.5,"399,490",1.6%
0.5,"393,068",1.6%


In [5]:
import pandas as pd
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import GridSearchCV, train_test_split
from collections import defaultdict
import pickle

In [7]:
!pip install surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 3.7 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp310-cp310-linux_x86_64.whl size=2799415 sha256=1ff1125d8dd2d1b96112aa86fef032045a814d3cc4bef3e15e877c55d6335db0
  Stored in directory: /root/.cache/pip/wheels/4b/3f/df/6acbf0a40397d9bf3ff97f582cc22fb9ce66adde75bc71fd54
Successfully built scikit-surprise

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
# Define the rating scale (here we assume ratings range from 0.5 to 5.0)

rating_min = movies_and_ratings['rating'].min()
rating_max = movies_and_ratings['rating'].max()

reader = Reader(rating_scale=(rating_min, rating_max))

data = Dataset.load_from_df(movies_and_ratings[['userId', 'movieId', 'rating']], reader)

In [7]:
print(movies_and_ratings.columns)

Index(['userId', 'movieId', 'rating', 'timestamp', 'title', 'genres'], dtype='object')


In [8]:
trainset, testset = train_test_split(data, test_size=0.25, random_state=42)

In [ ]:
param_grid = {
    'n_factors': [50, 100, 150],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.05]
}

# Use 3-fold cross-validation to search for the best parameters based on RMSE.
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(data)
 
print("Best RMSE score from grid search:", gs.best_score['rmse'])
print("Best parameters:", gs.best_params['rmse'])

In [ ]:
# Generating Top-N Recommendations 
def get_top_n(predictions, n=10):
    """Return the top-N recommendation for each user from a set of predictions."""
    top_n = defaultdict(list)
    # Aggregate predictions for each user.
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    # Sort the predictions for each user and retrieve the n highest ones.
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

In [ ]:
# Model Building and Training 

# Build the final model using the best parameters.
best_svd = gs.best_estimator['rmse']
best_svd.fit(trainset)

# Evaluate the model on the test set.
predictions = best_svd.test(testset)
test_rmse = accuracy.rmse(predictions)


In [ ]:
# Generate top-5 recommendations for each user in the test set.
top_n_recs = get_top_n(predictions, n=5)
 
movie_titles = movies_and_ratings[['movieId', 'title']].drop_duplicates().set_index('movieId')['title'].to_dict()

# Display recommendations for a sample user (e.g., userId = 1)
sample_user = 1
print(f"\nTop-5 recommendations for user {sample_user}:")
if sample_user in top_n_recs:
    for movie_id, est_rating in top_n_recs[sample_user]:
        title = movie_titles.get(int(movie_id), "Unknown Title")
        print(f"Movie ID: {movie_id}, Title: {title}, Predicted Rating: {est_rating:.2f}")
else:
    print("No recommendations available for this user.")